# Dataset management for UNET
- In this, we will use only the 50% (25k) target images of the dataset.

In [8]:
pip install opencv-python

In [10]:
!wget https://huggingface.co/datasets/fusing/fill50k/resolve/main/images.zip
!unzip images.zip
!rm images.zip

Streaming output truncated to the last 5000 lines.
  inflating: images/45164.png        
  inflating: images/3468.png         
  inflating: images/44408.png        
  inflating: images/42760.png        
  inflating: images/45431.png        
  inflating: images/7607.png         
  inflating: images/25051.png        
  inflating: images/33164.png        
  inflating: images/23211.png        
  inflating: images/29849.png        
  inflating: images/46520.png        
  inflating: images/6969.png         
  inflating: images/43511.png        
  inflating: images/8776.png         
  inflating: images/44598.png        
  inflating: images/8711.png         
  inflating: images/7442.png         
  inflating: images/23714.png        
  inflating: images/35138.png        
  inflating: images/24725.png        
  inflating: images/803.png          
  inflating: images/5287.png         
  inflating: images/26535.png        
  inflating: images/7861.png         
  inflating: images/24858.png        

In [3]:
import cv2
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torchvision import transforms
import math
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
%matplotlib inline

In [ ]:
hidden_dim = 32
num_groups = 8 # Splits the N channels into this number
T = 300
batch_size = 16
dataset_size = 1000

In [5]:
def load_images(folder):
    images = []
    for filename in range(dataset_size):
        img = cv2.imread(os.path.join(folder, str(filename) + '.png'))
        if img is not None:
            images.append(img)
    return images

imgArr = load_images('images')
imgArr = (torch.tensor(np.array(imgArr)).permute(0, 3, 1, 2).float() / 127.5) - 1
imgArr = transforms.Resize((64, 64))(imgArr)

dataset = TensorDataset(imgArr)
loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# Time Embedding and Residual block

In [6]:
# Time step embedding
#t=0 = least noisy, t=T-1

class TimestepMLP(nn.Module):
    # A small two-layer MLP (Linear → SiLU → Linear) that takes the raw sinusoidal embedding vector and projects it into a learned representation.
    # Input size is sinusoidal_dim, output size is hidden_dim.
    # This is the "learned" part of the timestep embedding — the sinusoidal part itself has no learned parameters.
    def __init__(self, sinusoidal_dim, hidden_dim):
        super().__init__()
        self.fc1 = nn.Linear(sinusoidal_dim, hidden_dim)
        self.act = nn.SiLU()
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, t_emb):
        # t_emb: (batch, sinusoidal_dim) — output of the sinusoidal function
        x = self.fc1(t_emb)
        x = self.act(x)
        x = self.fc2(x)
        return x  # (batch, hidden_dim)

class TimeEmbedding(nn.Module):
    # The full timestep embedding module. Takes a batch of integer timesteps t (shape (batch,), ranging 0 to T-1)
    # and returns a learned embedding vector per timestep (shape (batch, hidden_dim)).
    # Internally: computes a fixed sinusoidal embedding (sin/cos at different frequencies, same family as transformer positional embeddings),
    # then passes it through TimestepMLP to produce the final learned representation.
    # Instantiated once in the UNet, called once per forward pass, output shared across all residual blocks.
    def __init__(self, sine_dim):
        super().__init__()
        self.sine_dim = sine_dim
        self.mlp = TimestepMLP(sine_dim, hidden_dim) # sine_dim = 8, hidden_dim = 32; better to have hidden_dim > sine_dim. hidden_dim is the output of the MLP

    def sinusoidal_embd(self, t):
        half_dim = self.sine_dim // 2
        freqs = torch.exp(-math.log(10000) * torch.arange(half_dim, device=t.device) / half_dim) # [half_dim] - 1D
        args = t[:, None].float() * freqs[None, :] # t * freqs; just broadcasting to multiply different shapes
        return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)

    def forward(self, t):
        t_sin = self.sinusoidal_embd(t)   # (batch, sinusoidal_dim)
        return self.mlp(t_sin)                  # (batch, hidden_dim)

feature_map = the image at current timestep

In [7]:
# Residual block

class ResidualBlock(nn.Module):
    # The core building block of the UNet. Takes an image feature map x (shape (batch, in_channels, H, W)) and
    # the timestep embedding t_emb (shape (batch, hidden_dim)) and returns a new feature map (shape (batch, out_channels, H, W)).
    # Internally: two Conv2d 3x3 layers with GroupNorm and SiLU activation, with the timestep embedding injected
    # (via a small linear projection + unsqueeze) between the two convs. A skip connection (x + f(x)) wraps the whole thing
    # — uses a 1x1 Conv2d to match channels if in_channels != out_channels, otherwise Identity.
    def __init__(self, in_channels, out_channels, hidden_dim):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.norm1 = nn.GroupNorm(num_groups, out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.norm2 = nn.GroupNorm(num_groups, out_channels)
        self.act = nn.SiLU()
        self.t_proj = nn.Linear(hidden_dim, out_channels)
        self.skip = nn.Conv2d(in_channels, out_channels, kernel_size=1) if in_channels != out_channels else nn.Identity()

    def projTimeEmbd(self, t_emb):
        return self.t_proj(t_emb).unsqueeze(-1).unsqueeze(-1)

    def forward(self, x, t_emb):
        # first conv
        y = self.conv1(x)
        y = self.norm1(y)
        y = self.act(y)
        # inject timestep
        y = y + self.projTimeEmbd(t_emb)
        # second conv
        y = self.conv2(y)
        y = self.norm2(y)
        # skip connection
        y = y + self.skip(x)
        return y


In [8]:
# block = ResidualBlock(in_channels=3, out_channels=32, hidden_dim=32)
# x = torch.randn(4, 3, 64, 64)       # batch=4, 3 channels, 64x64
# t = torch.randint(0, 300, (4,))
# te = TimeEmbedding(sine_dim=8)
# t_emb = te(t)                        # (4, 32)
# out = block(x, t_emb)
# print(out.shape)                     # expect (4, 32, 64, 64)

# UNet
The UNet is purely a pattern recogniser — "given this noisy image at this noise level, what noise pattern do I see?"

In [9]:
# downsample block

class DownSample(nn.Module):
    # Halves the spatial resolution of the feature map while increasing channel count. A single Conv2d with stride=2
    # (learnable downsampling, preferred over pooling in diffusion UNets since the kernel weights are trained).
    # Takes (batch, in_channels, H, W) → outputs (batch, out_channels, H/2, W/2).
    # No timestep embedding needed — purely a spatial operation.
    # Used at the end of each contracting stage in the UNet.
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.downsample = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, stride=2)

    def forward(self, x):
        return self.downsample(x)

In [10]:
# ds = DownSample(32, 64)
# x = torch.randn(4, 32, 64, 64)
# out = ds(x)
# print(out.shape) # expect (4, 64, 32, 32) — channels 32→64, spatial 64→32

In [11]:
# upsample block

class UpSample(nn.Module):
    # Doubles the spatial resolution while decreasing channel count, then concatenates the skip connection from the matching contracting stage.
    # Uses ConvTranspose2d with kernel_size=4, stride=2, padding=1 for exact 2× upsampling (no off-by-one ambiguity).
    # Takes (batch, in_channels, H, W) + skip (batch, skip_channels, H*2, W*2) → outputs (batch, out_channels + skip_channels, H*2, W*2).
    # The subsequent ResidualBlock in the expanding path receives this concatenated tensor as its in_channels.
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.upsample = nn.ConvTranspose2d(in_channels, out_channels, kernel_size=4, stride=2, padding=1)
        # kernel_size=4, stride=2, padding=1 is the standard combination for exact 2× upsampling
        # — produces output exactly double the input spatial size, no off-by-one ambiguity.
    def forward(self, x, skip):
        x = self.upsample(x)
        x = torch.cat([x, skip], dim=1)  # concatenate skip connection along channel dim
        return x

In [12]:
# us = UpSample(64, 32)
# x = torch.randn(4, 64, 32, 32)
# skip = torch.randn(4, 32, 64, 64)  # from matching contracting stage
# out = us(x, skip)
# print(out.shape)  # expect (4, 64, 64, 64) — 32 up-channels + 32 skip-channels

channels and image resolutions are different things

In [13]:
class UNet(nn.Module):
    def __init__(self):
        super().__init__()

        # 1. TimeEmbedding
        self.te = TimeEmbedding(sine_dim=8)

        # 2. Contracting path
        # stage 1: 3→32 channels, 64×64
        self.resDn1 = ResidualBlock(in_channels=3, out_channels=32, hidden_dim=32)
        self.ds1 = DownSample(in_channels=32, out_channels=64)
        # stage 2: 64 channels, 32×32
        self.resDn2 = ResidualBlock(in_channels=64, out_channels=64, hidden_dim=32)
        self.ds2 = DownSample(in_channels=64, out_channels=128)
        # stage 3: 128 channels, 16×16
        self.resDn3 = ResidualBlock(in_channels=128, out_channels=128, hidden_dim=32)
        self.ds3 = DownSample(in_channels=128, out_channels=256)

        # 3. Bottleneck: 256 channels, 8×8
        self.resBottle = ResidualBlock(in_channels=256, out_channels=256, hidden_dim=32)

        # 4. Expanding path
        # stage 1: upsample 256→128, concat skip(128) → 256 in for resblock
        self.us1 = UpSample(in_channels=256, out_channels=128)
        self.resUp1 = ResidualBlock(in_channels=256, out_channels=128, hidden_dim=32)
        # stage 2: upsample 128→64, concat skip(64) → 128 in for resblock
        self.us2 = UpSample(in_channels=128, out_channels=64)
        self.resUp2 = ResidualBlock(in_channels=128, out_channels=64, hidden_dim=32)
        # stage 3: upsample 64→32, concat skip(32) → 64 in for resblock
        self.us3 = UpSample(in_channels=64, out_channels=32)
        self.resUp3 = ResidualBlock(in_channels=64, out_channels=32, hidden_dim=32)

        # 5. Final conv: 32→3 channels
        self.final_conv = nn.Conv2d(32, 3, kernel_size=1)

    def forward(self, x, t):
        skipconn = []

        # 1. compute t_emb from TimeEmbedding
        t_emb = self.te(t)
        # 2. contracting path — save each stage's output as skip connection
        dn1 = self.resDn1(x, t_emb)
        skipconn.append(dn1)
        dn1 = self.ds1(dn1)

        dn2 = self.resDn2(dn1, t_emb)
        skipconn.append(dn2)
        dn2 = self.ds2(dn2)

        dn3 = self.resDn3(dn2, t_emb)
        skipconn.append(dn3)
        dn3 = self.ds3(dn3)

        # 3. bottleneck
        rsbttle = self.resBottle(dn3, t_emb)
        # 4. expanding path — pass skip connections into each UpSample
        up1 = self.us1(rsbttle, skipconn[2])
        up1 = self.resUp1(up1, t_emb)

        up2 = self.us2(up1, skipconn[1])
        up2 = self.resUp2(up2, t_emb)

        up3 = self.us3(up2, skipconn[0])
        up3 = self.resUp3(up3, t_emb)

        # 5. final conv
        return self.final_conv(up3)

In [14]:
# model = UNet()
# x = torch.randn(4, 3, 64, 64)
# t = torch.randint(0, 300, (4,))
# out = model(x, t)
# print(out.shape)  # expect (4, 3, 64, 64)

# Noise Scheduler
The noise scheduler precomputes a table of values for every timestep t from 0 to T-1, so during training you can instantly look up "how much noise belongs at step t" without recomputing anything.
- beta_t      — how much noise to add at this specific step (small number, e.g. 0.0001 to 0.02)
- alpha_t     — how much of the signal remains = 1 - beta_t
- alpha_bar_t — cumulative product of all alphas up to t = α₁ × α₂ × ... × αₜ

alpha_bar_t is the important one — it's what lets you jump directly to "noisy image at step t" in one shot during training, without iterating through all previous steps.

* noisy_image = sqrt(alpha_bar_t) * clean_image + sqrt(1 - alpha_bar_t) * noise

Where noise is random Gaussian noise torch.randn_like(image). As t increases, alpha_bar_t shrinks toward 0 (less clean image) and 1 - alpha_bar_t grows toward 1 (more noise).

In [15]:
class NoiseScheduler:
    def __init__(self, T=300, beta_start=0.0001, beta_end=0.01):
        self.T = T
        # linearly spaced betas from small to large
        self.beta_t = torch.linspace(beta_start, beta_end, T)
        self.alpha_t = 1 - self.beta_t
        self.alpha_bar_t = torch.cumprod(self.alpha_t, dim=0)

    def add_noise(self, x, t, noise):
        # x: (batch, 3, 64, 64) clean image
        # t: (batch,) integer timesteps
        # noise: (batch, 3, 64, 64) random gaussian
        ab = self.alpha_bar_t[t] # (batchl, )
        a = ab.view( -1, 1, 1, 1)
        noisy_image = torch.sqrt(a) * x + torch.sqrt(1 - a) * noise # noise = torch.randn_like(x) # (batch, 3, 64, 64)—random values from N(0,1)
        return noisy_image


# Training loop

1. sample random t per image
2. sample random noise
3. call add_noise
4. pass noisy image + t into UNet
5. MSE between predicted noise and actual noise

In [ ]:
model = UNet()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
scheduler = NoiseScheduler(T=300, beta_start=0.0001, beta_end=0.02)
losses = []

# training loop
for epoch in range(100):
    for (x,) in loader:
        t = torch.randint(0, 300, (x.shape[0],))
        noise = torch.randn_like(x)
        noisy_img = scheduler.add_noise(x, t, noise)
        pred_noise = model(noisy_img, t)
        loss = F.mse_loss(pred_noise, noise)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        losses.append(loss.item())

    print(f'epoch {epoch} loss {loss.item():.4f}')

In [ ]:
plt.plot(losses)
plt.xlabel('step')
plt.ylabel('loss')
plt.title('training loss')
plt.show()